# 第20章 本当に良くなった？：評価と統計

「なんとなく良くなった」を、数字と信頼区間で言い切る

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：ボンゴ先生LoRAを、①パープレキシティ ②自動チェック ③4択クイズ（忘れていないか） ④LLMによる審査 の4つの方法で評価し、ブートストラップで信頼区間をつけた評価レポート
- **身につく数学・理論**：評価の種類（自動指標・ベンチマーク・LLM審査・人間評価） ／ 保留データでのパープレキシティ ／ 選択式ベンチマークを対数尤度で解く方法 ／ LLM-as-a-judge と位置の偏り ／ 平均・標準誤差・ブートストラップ信頼区間 ／ データ漏洩（コンタミネーション）とグッドハートの法則
- **新しく使うPython・PyTorch**：`random.Random` と `randrange` で復元抽出 ／ `.sort()` で並べて95%の範囲を取り出す ／ f文字列で表を作る

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> 第18〜19章で、モデルをボンゴ先生に変身させました。でも、「本当に良くなったの？」「元の賢さを忘れていない？」と聞かれたら、何と答えますか？ この章では、4種類の評価方法でモデルを測り、**統計**を使って「その差は偶然じゃない」と言えるかどうかを判断します。研究の世界で一番信頼されるのは、派手なデモではなく、誠実な評価です。

> **🔗 これまでの復習とつながり：第5章・第10章・第18章とのつながり**
>
> - **第5章（パープレキシティと検証データ）**：訓練に使っていないデータ上で損失とパープレキシティ（$e^{\text{損失}}$）を測ることで汎化性能を確かめる考え方を学びました。
> - **第10章（平均と標準偏差）**：数字の「真ん中（平均）」だけでなく「どれくらいばらつくか（分散・標準偏差）」を測る発想を、ここでは標本サイズの揺らぎ（標準誤差・信頼区間）へ広げます。
> - **第18章（壊滅的忘却）**：ボンゴ先生の口調を学ばせた結果、元の一般知識を忘れていないか、この章の4択クイズと統計で実際に検証します！

**ミナト**：いくつか質問して「いい感じ！」じゃダメなんですか？

**ボンゴ先生**：ダメとは言わないが、それは**感想**だ。キミが試した3問が、たまたまうまくいっただけかもしれない。人間は、自分が作ったものを良く見たいものなんだ。私も、自分のダジャレは全部面白いと思っている。だから、自分をだまさない仕組みが要る。ファインマンも言っている。「第一の原則は、自分をだまさないこと。そして自分は、いちばんだまされやすい人間なのだ」と。

## 準備：比べる2つのモデル

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：無料GPU（T4）推奨です。ベースラインとボンゴLoRAの2モデルを扱うためVRAM約5〜7GBを使います。
> - **ダウンロードと所要時間**：初回に `Qwen2.5-0.5B-Instruct`（約1GB）と評価データ3件を取得します。GPUなら学習から4種類の評価完了まで約2〜3分です。
> - **GPUが使えない日の代替**：ブートストラップ統計と評価レポートのセルは**CPUだけで一瞬**で動きます。CPUのみで全セルを試す場合は、学習を1エポック・評価件数を `heldout_rows[:4]`、`test_questions[:4]`、`quiz[:5]` に絞ると待ち時間を減らせます。
> - **ライブラリ互換性**：`dtype=` は `transformers >= 4.56` 対応（推奨環境は `requirements-book.txt` を参照）。

評価には「比べる相手」が必要です。元のQwen2.5-0.5B-Instruct（**ベースライン**）と、第19章と同じ方法で作るボンゴ先生LoRAを用意します。ただし今回は、データの**最後の12問を評価用に取り分けて**から学習させます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!pip install -q peft trl
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_test.jsonl
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/mini_quiz.jsonl
import json, math, os, random
for fname in ("bongo_sft.jsonl", "bongo_test.jsonl", "mini_quiz.jsonl"):
    if not os.path.exists(fname):
        raise RuntimeError(f"{fname} のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

rows = [json.loads(l) for l in open("bongo_sft.jsonl", encoding="utf-8")]
train_rows, heldout_rows = rows[:48], rows[48:]          # 最後の12問は学習に使わない
test_questions = [json.loads(l)["question"] for l in open("bongo_test.jsonl", encoding="utf-8")]
quiz = [json.loads(l) for l in open("mini_quiz.jsonl", encoding="utf-8")]
print(len(train_rows), len(heldout_rows), len(test_questions), len(quiz))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
48 12 12 20
```

</details>

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
name = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(name)
baseline = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device).eval()

ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["question"]}],
                         "completion": [{"role": "assistant", "content": r["answer"]}]} for r in train_rows])
trainer = SFTTrainer(
    model=AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32), train_dataset=ds, processing_class=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"),
    args=SFTConfig(output_dir="eval-bongo", num_train_epochs=3, per_device_train_batch_size=4, learning_rate=2e-4,
                   report_to="none", save_strategy="no", logging_steps=100, bf16=False, fp16=torch.cuda.is_available()))
trainer.train()
bongo = trainer.model.merge_and_unload().to(device).eval()
print("準備完了")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
{'train_runtime': 31.2, 'train_loss': 1.74, 'epoch': 3.0}
準備完了
```

</details>

#### 📝 確認問題（`ch20-q01` · Lv.1 · ベースライン比較と学習前のデータ分割（保留データ）の鉄則）

モデルを評価するときに「学習前のベースライン」を用意し、さらに評価用の保留データ（`heldout_rows`）を**学習を始める前**に取り分けておく理由はどれですか？

- **a.** ベースラインと保留データを先に登録しないと、TRLの `SFTTrainer` が仕様上の構文エラーで停止するから
- **b.** 学習完了後にモデルが最も上手に答えられた12問を保留データとして事後選定したほうが公平に比較できるから
- **c.** 学習前に保留データを多く取り分けておくほど、訓練ループのエポック数が自動的に増えて収束が早まるから
- **d.** ベースラインがないと改善幅が測れず、学習後に選ぶと訓練問題が混ざり丸暗記を実力と誤認するから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：科学的な比較実験における「比較対象（対照群）」と「未知テストの事前隔離」の2つの原則を思い出しましょう。
- **ヒント 2**：学習前の基準点（ベースライン）と、訓練に一切見せない保留データの分割タイミングがなぜ重要か確認しましょう。
- **正解**：`d`

**考え方**：誠実な評価の基本は「同じ条件でベースラインと比較すること」と「学習前に一度もモデルに見せない保留データを隔離すること」です。

1. ベースライン（元のモデル）を用意 → 改善前後の差分を測れる。
1. 学習前に `rows[:48]` と `rows[48:]` に分割 → 保留12問が訓練に漏れるのを防ぐ。
1. 同じ評価関数に両モデルを通して公平に比較する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q01

</details>

## 評価1：保留データのパープレキシティ

学習に使わなかった12問の「ボンゴ先生の答え」を、それぞれのモデルがどれくらい**予想できるか**を測ります。第5章のパープレキシティです。答えの部分だけの損失を計算します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def answer_nll_sum(model, question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + a]).to(device)
    labels = torch.tensor([[-100] * len(p) + a]).to(device)
    mean_nll = model(input_ids=ids, labels=labels).loss.item()
    return mean_nll * len(a), len(a)

for label, m in [("ベースライン", baseline), ("ボンゴLoRA", bongo)]:
    pairs = [answer_nll_sum(m, r["question"], r["answer"]) for r in heldout_rows]
    mean = sum(s for s, _ in pairs) / sum(n for _, n in pairs)   # 全答えトークン数で重みづけしたコーパス平均NLL
    print(f"{label}：損失 {mean:.3f}  パープレキシティ {math.exp(mean):.1f}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ベースライン：損失 2.951  パープレキシティ 19.1
ボンゴLoRA：損失 2.214  パープレキシティ 9.2
```

</details>

ボンゴLoRAのほうが、見たことのないボンゴ先生の答えを、ずっと「ありそう」だと予想できています。口調を学べた証拠です。なお、答えの長さが問題ごとに違うため、1問ごとの平均をそのまま平均するのではなく、**全トークンの損失合計 ÷ 全トークン数**（トークン重みづけのコーパス平均NLL）からパープレキシティを求めています。

> **💡 理論のポイント：保留データ**
>
> 評価には、学習に**一度も使っていない**データ（**保留データ**、held-out data）を使います。学習に使ったデータで測ると、丸暗記しているだけでも良い数字が出てしまうからです（第5章の過去問の話）。今回は、データを作った段階で最後の12問を取り分けておきました。データを分けるのは、学習を始める**前**にやるのが鉄則です。

> **⚠️ つまずき注意：パープレキシティを比べるときの3つの注意点**
>
> 1. **トークナイザーが違うモデル同士は直接比べられない**（第5章・第14章）：語彙サイズや1文字あたりのトークン数が違うと「1トークンを当てる難しさ」が別物になるため、そのままパープレキシティを並べて優劣をつけてはいけません（比較するには第14章の「1文字／1バイトあたりの損失」に換算します）。
> 2. **評価データセットや前処理が違う数字は比べられない**：文章のジャンル・長さ・改行や記号の前処理が違うだけでパープレキシティは大きく変わるため、必ず**まったく同じ保留データと前処理**で比較します。
> 3. **パープレキシティが低い ≠ 指示追従や安全性が高い**：パープレキシティは「お手本に似た文章を出す確率」しか測れません。事実の正確さ・指示の遵守・安全に断る力は、この先の自動チェック・選択式ベンチマーク・LLM審査・人間評価と組み合わせて確かめる必要があります。

#### 📝 確認問題（`ch20-q02` · Lv.2 · 保留データでパープレキシティを測る理由と数値の読み方）

学習に使ったデータではなく、**保留データ（held-out data）** を使って評価しなければならない理由はどれですか？

- **a.** 未知の保留データを使うほうが常にパープレキシティが低く算出され、レポートの見栄えが良くなるから
- **b.** 学習済みデータでは交差エントロピーが0になるため、数式上パープレキシティが定義できないから
- **c.** 保留データで推論すると勾配計算が自動的に省かれ、評価時のGPU消費電力がちょうど半分に下がるから
- **d.** 学習データでは丸暗記でも好成績になり、未知の質問への本当の実力（汎化性能）を測れないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：学校の試験で、前日に丸暗記した過去問とまったく同じ問題だけが出たとき、本当の実力が測れるか考えてみましょう。
- **ヒント 2**：訓練データ上の損失は過学習（丸暗記）でも下がり続けます。未知の質問への汎化を確認するにはどのデータを使うべきでしょうか？
- **正解**：`d`

**考え方**：モデルが訓練データ60問を丸暗記していても、訓練損失はどんどん下がります。一度も見ていない保留データで測ることで初めて「ボンゴ先生の口調が新しい質問にも一般化したか」を証明できます。

1. 訓練データ上のスコア：丸暗記でも良くなる。
1. 保留データ上のスコア：未知の文を予測できるか（汎化）を測る。
1. ベースライン 19.1 → ボンゴLoRA 9.2 と大きく下がったことが、口調習得の客観的証拠になる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q02

</details>

## 評価2：自動チェック（ルールで数える）

ボンゴ先生らしさには、ルールで確かめられる特徴があります。「感嘆符（！）を使う」「長すぎない」「きちんと止まる（`<|im_end|>` を出す）」。評価用の12問に答えさせて、数えてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def generate(model, question, n=120):
    inputs = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True), (new == tok.convert_tokens_to_ids("<|im_end|>")).any().item()

def style_checks(text, stopped):
    return {"！を使う": "！" in text, "150字以内": len(text) <= 150, "きちんと止まる": stopped}

answers = {}
for label, m in [("ベースライン", baseline), ("ボンゴLoRA", bongo)]:
    answers[label] = [generate(m, q) for q in test_questions]
    results = [style_checks(t, s) for t, s in answers[label]]
    summary = "  ".join(f"{k} {sum(r[k] for r in results)}/{len(results)}" for k in results[0])
    print(f"{label}：{summary}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ベースライン：！を使う 1/12  150字以内 3/12  きちんと止まる 9/12
ボンゴLoRA：！を使う 12/12  150字以内 11/12  きちんと止まる 12/12
```

</details>

> **💡 理論のポイント：自動チェックの長所と短所**
>
> ルールで数える評価は、**速く、安く、何度でも同じ結果**が出ます。ソフトウェアのテストと同じで、変更のたびに自動で回せます。ただし、「！を使う」はボンゴ先生らしさのごく一部で、**内容が正しいか**は分かりません。「！」を付けるだけの役立たずなモデルも満点になります。1つの指標だけを信じないことが大切です。

#### 📝 確認問題（`ch20-q03` · Lv.2 · ルールベースの自動チェックの長所と限界）

「！を使う」「150字以内」「`<|im_end|>` で止まる」といったルールベースの自動チェックについて、最も適切な評価姿勢はどれですか？

- **a.** 「！」の使用率が12/12になれば、回答内容の事実正確性や分かりやすさも100%保証されたと言える
- **b.** ルール判定は実行ごとに乱数で結果が揺れるため、同じ出力に対して再現性のある検証ができない
- **c.** 高速・安価・再現性100%で形式確認に最適だが、内容の正しさは測れないため他評価と組み合わせる
- **d.** ルールベースの評価は単純すぎて実務の役に立たないため、自動チェックは廃止して人間評価だけ行うべき

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ソフトウェア開発の単体テストに例えて、ルールベース自動チェックが「得意なこと」と「絶対に見抜けないこと」を整理しましょう。
- **ヒント 2**：終了トークンや文字数制限などの形式崩れは高速かつ決定的に検知できますが、「！！！」だけの壊れた回答を防ぐには何が必要でしょうか？
- **正解**：`c`

**考え方**：ルールベース自動チェックは「明確なフォーマット要件・停止条件の回帰テスト」に絶大な威力を発揮しますが、意味的品質は別指標（ベンチマーク・LLM審査・人間確認）で補う必要があります。

1. 長所：速い・無料・100%決定的で毎回CIのように回せる。
1. 短所：「！だけ出力」などハック可能で、事実性や分かりやすさは見抜けない。
1. 結論：複数評価の1本目の柱として使う。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q03

</details>

## 評価3：4択クイズ（忘れていないか）

ボンゴ先生化で、元の知識を忘れていないか（第18章の壊滅的忘却）を確かめます。20問の4択クイズを用意しました。LLMに4択を解かせるには、**各選択肢の確率**を比べる方法がよく使われます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def choice_logprob(model, question, choice):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(choice, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + c]).to(device)
    logprobs = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
    targets = ids[0, 1:]
    return logprobs[len(p) - 1:, :].gather(1, targets[len(p) - 1:, None]).sum().item()

def quiz_score(model):
    correct = []
    for item in quiz:
        scores = [choice_logprob(model, item["question"], c) for c in item["choices"]]
        correct.append(int(scores.index(max(scores)) == item["answer"]))
    return correct

quiz_results = {label: quiz_score(m) for label, m in [("ベースライン", baseline), ("ボンゴLoRA", bongo)]}
for label, c in quiz_results.items():
    print(f"{label}：{sum(c)}/{len(c)} 問正解")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ベースライン：17/20 問正解
ボンゴLoRA：16/20 問正解
```

</details>

> **🐍 Python・PyTorchのポイント：log_softmax と gather**
>
> `torch.log_softmax(…)` は、softmaxで確率にしてから、そのlogをとる計算を一度にやる関数です（別々にやるより、数値の誤差が出にくい）。<br>
> `logprobs[…].gather(1, targets[…, None])` は、「各行（各位置）から、`targets` で指定した列（実際に来たトークン）の値だけを拾う」操作です。第8章の `probs[torch.arange(len(y)), y]` と同じことを、別の書き方でしています。拾った「各トークンの対数確率」を `.sum()` で足せば、選択肢の文字列全体の対数確率になります（第5章：確率の掛け算は、対数の足し算）。

> **💡 理論のポイント：選択肢の確率で解く**
>
> 「次のトークン予測」しかできないLLMに4択を解かせるには、質問の後に各選択肢を続けたとき、**その選択肢の文字列全体の確率**（対数の合計）を計算し、一番高いものを答えとします。生成させて「答えは2番」と書かせる方法より、ぶれが少なく、小さなモデルでも測れます。有名なベンチマーク（MMLUや、その日本語版のJMMLUなど）でも、この方法がよく使われます。
>
> 選択肢の長さが違うと、長いほど確率の合計が小さくなって不利になるので、長さで割る（1トークンあたりにする）こともあります。どちらの方式で測ったかは、結果と一緒に必ず書いておきましょう。

<details><summary><b>基礎の補足：対数確率（log-prob）の大小関係と長さ正規化</b>（クリックで開く）</summary>

確率 $p$ は $0 < p \le 1$ なので、その対数 $\log p$ は**必ず0以下（マイナス）**になります。
- たとえば `A: -4.8`、`B: -1.5`、`C: -2.9`、`D: -6.1` なら、最も0に近い最大値 **`B (-1.5)`** が最高確率（$e^{-1.5} \approx 22.3\%$）です。
- 選択肢が長いと負の数（$\log p_t \le 0$）を何度も足すことになるため不利になりやすい点に注意します。

```python no-run
# 1トークンあたりの平均対数確率（長さ正規化）の例
raw_scores = [-2.0, -3.0]     # 合計では選択肢0 (-2.0) が勝ち
token_lens = [1, 3]           # でも選択肢1は3トークンある
norm_scores = [s / n for s, n in zip(raw_scores, token_lens)]  # [-2.0, -1.0] → 選択肢1が逆転！
```

</details>

#### 📝 確認問題（`ch20-q04` · Lv.2 · 4択ベンチマークを選択肢の対数確率（logprob）で評価する仕組み）

4択クイズをLLMに解かせるとき、各選択肢のトークンごとの対数確率を合計した `choice_logprob` がそれぞれ `A: -4.8`, `B: -1.5`, `C: -2.9`, `D: -6.1` でした。モデルが最も「ありそう」と判断した選択肢はどれですか？また、対数確率の合計で比べる際の注意点はどれですか？

- **a.** 0に最も近い最大値 **B（`-1.5`）** が選ばれる。長い選択肢ほど負の和が増え不利なため長さ正規化も使う
- **b.** 選択肢の対数確率はトークンごとにすべて掛け算して合算しないと、選択肢全体の確率スコアにならない
- **c.** 対数確率はすべて負の値になるため、いったん絶対値で正の数へ変換してから最大値を持つ選択肢を選ぶ
- **d.** 絶対値が最大の **D（`-6.1`）** が選ばれる。長い選択肢ほどトークン数が増えて確率の和が大きくなり有利になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：確率 $p \in (0, 1]$ の対数 $\log p$ は0以下の値をとり、確率が高いほど `0` に近い（大きい）値になります。
- **ヒント 2**：$e^{-1.5} \approx 0.223$ と $e^{-6.1} \approx 0.0022$ を比べましょう。また負の値をトークン数だけ足すと長い選択肢がどう影響を受けるかも確認しましょう。
- **正解**：`a`

**考え方**：確率 $0 < p \le 1$ に対して $\log p \le 0$ であり、最大値（最も0に近い値）が高確率を意味します。またトークンを1個足すたびに負の値が加算されるため、選択肢の長さに差があるベンチマークでは1トークンあたり平均対数確率（長さ正規化）がよく使われます。

1. 確率の積 $\prod_t p_t$ の対数は和 $\sum_t \log p_t$（すべて $\le 0$）。
1. `max([-4.8, -1.5, -2.9, -6.1]) = -1.5` → 選択肢 B が最高確率。
1. 選択肢の長さが不ぞろいなときは `sum_logprob / len(choice_tokens)` で長さの有利・不利を補正する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q04

</details>

#### 📝 確認問題（`ch20-q05` · Lv.2 · 対数確率の最大選択と長さ正規化による順位逆転の予測）

4つの選択肢の対数確率スコアからモデルの予測インデックスを選ぶ次のコードは何を表示しますか？

```python
scores = [-4.2, -1.8, -3.1, -5.0]
best = scores.index(max(scores))
print(best, scores[best])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：負の数どうしを比較すると、絶対値が小さく `0` に近い値ほど大きくなります。Pythonのリストインデックスは0始まりです。
- **ヒント 2**：`[-4.2, -1.8, -3.1, -5.0]` の中で最も大きい値を見つけ、その0始まりの位置番号と値を並べて答えましょう。
- **期待される出力**：`1 -1.8`

**考え方**：`max([-4.2, -1.8, -3.1, -5.0])` は `-1.8`、そのインデックスは `1` なので `1 -1.8` が出力されます。

1. 大小比較：`-1.8 > -3.1 > -4.2 > -5.0`。
1. `max(scores) = -1.8`、インデックスは `1`。
1. 出力：`1 -1.8`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q05

</details>

17問と16問。ボンゴLoRAは1問減りました。……これは「忘れた」のでしょうか？ それとも偶然の誤差でしょうか？

## 統計：その差は偶然か？

20問は、ほんの一部の問題です。別の20問を選んでいたら、結果は逆になったかもしれません。そこで、**ブートストラップ**という方法で、「もし問題を選び直したら、差はどれくらいばらつくか」を調べます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def bootstrap_diff(a, b, n_boot=10000, seed=0):
    """a, b：同じ問題に対する正解（1）・不正解（0）のリスト。差（b - a）の95%信頼区間を返す"""
    rng = random.Random(seed)
    n = len(a)
    diffs = []
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]          # 問題を「復元抽出」で選び直す
        diffs.append(sum(b[i] - a[i] for i in idx) / n)
    diffs.sort()
    return sum(b[i] - a[i] for i in range(n)) / n, diffs[int(0.025 * n_boot)], diffs[int(0.975 * n_boot)]

base_c = [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1]     # 例：17/20
bongo_c = [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1]    # 例：16/20（得意・不得意な問題が違う）
diff, low, high = bootstrap_diff(base_c, bongo_c)
print(f"正答率の差：{diff:+.2f}  95%信頼区間：[{low:+.2f}, {high:+.2f}]")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
正答率の差：-0.05  95%信頼区間：[-0.25, +0.15]
```

</details>

> **💡 理論のポイント：ブートストラップと信頼区間**
>
> **ブートストラップ**（bootstrap）は、手元のデータから「同じ数だけ、重複を許して選び直す」（**復元抽出**）ことを何千回もくり返し、結果のばらつきを調べる方法です。数式をほとんど使わずに、どんな指標の信頼区間も計算できる、とても便利な道具です。
>
> **95%信頼区間**は、ざっくり言うと「問題を選び直しても、差はたいていこの範囲に収まるだろう」という幅です。上の例では、ボンゴLoRAはベースラインが間違えた2問に正解し、ベースラインが正解した3問を間違えて、差し引き1問減っています。信頼区間は**0（差なし）をまたいでいる**ので、「ボンゴLoRAが知識を忘れた」とは**言い切れません**。20問は少なすぎるのです。言い切るには、問題を増やす（数百問）必要があります。
>
> この「同じ問題で両方のモデルを比べ、差をブートストラップする」方法を **対応のあるブートストラップ**（paired bootstrap）と言います。同じ問題で比べるので、問題の難しさのばらつきが打ち消され、少ない問題数でも差を検出しやすくなります。

#### 📐 数学のポイント：標準誤差・95%信頼区間・ブートストラップ：8ステップで理解する

**① 解決したい問題**：20問のクイズでベースラインが 17/20（85%）、ボンゴLoRAが 16/20（80%）。この「1問（5%）の差」は本当に忘れたのか、それとも問題選びの偶然かを見分けたい。

**② 素朴な比較が危険な理由**：正答率の数字だけを見ると「85% vs 80%」は、20問テストでも1,000問テストでも同じに見えてしまう。でも20問なら1問たまたま転ぶだけで±5%動く！

**③ 発想**：テスト問題 $n$ 問は「世の中にある無限の問題」からランダムに引いた標本とみなせる。標本サイズ $n$ に応じた「正答率のぶれ幅（**標準誤差 SE**）」を数式または復元抽出（**ブートストラップ**）で見積もり、**95%信頼区間**で表す。

**④ 記号表と数式**：

$$\text{SE} = \sqrt{\frac{\hat{p}(1 - \hat{p})}{n}}, \qquad \text{95\%信頼区間（Wald近似）} = \hat{p} \pm 1.96\,\text{SE}$$

| 記号 | 意味 | Pythonでの対応 |
|---|---|---|
| $n$ | 評価した問題数 | `len(results)` |
| $\hat{p}$ | 観測された正答率（$0 \sim 1$） | `sum(results) / n` |
| $\text{SE}$ | 標準誤差（問題を引き直したときの正答率のぶれの目安） | `math.sqrt(p * (1 - p) / n)` |
| $1.96\,\text{SE}$ | 95%信頼区間の片側の幅（正規分布の両側5%境界） | `1.96 * se` |

**⑤ 小さな問題数 vs 大きな問題数で手計算（$\hat{p} = 0.80$ のとき）**：
- **$n = 20$ 問のとき**：
  $$\text{SE} = \sqrt{\frac{0.80 \times 0.20}{20}} = \sqrt{0.008} \approx 0.0894 \implies 1.96\,\text{SE} \approx 0.175$$
  95%信頼区間は **$[0.625,\; 0.975]$**（幅 35%ポイント！）。1〜2問（5〜10%）の差は完全にノイズに埋もれます。
- **$n = 400$ 問のとき**：
  $$\text{SE} = \sqrt{\frac{0.80 \times 0.20}{400}} = \sqrt{0.0004} = 0.020 \implies 1.96\,\text{SE} \approx 0.0392$$
  95%信頼区間は **$[0.761,\; 0.839]$**（±3.9%）まで一気に引き締まります！

**⑥ $\frac{1}{\sqrt{n}}$ の法則**：分母が $\sqrt{n}$ なので、**誤差の幅を半分にするには問題数を $2^2 = 4$ 倍**、3分の1にするには **$3^2 = 9$ 倍** にする必要があります。評価の精度を上げるのは地道で大変なのです。

**⑦ ブートストラップとの関係**：上の式は1モデルの正答率向けですが、**2つのモデルの差**や複雑な指標では、上のコードのように手元データから同じ $n$ 問を**復元抽出**（重複OKで引き直し）して1万回やり直す **対応のあるブートストラップ** が強力です。同じ問題番号 `idx` で両モデルの差 `b[i] - a[i]` をとるため、問題の難易度のばらつきが打ち消されます。

**⑧ 理解のための最小Pythonコード（Wald 95%信頼区間）**：

```python no-run
import math
def wald_ci95(results):
    n = len(results)
    p = sum(results) / n
    se = math.sqrt(p * (1 - p) / n)
    return p, max(0.0, p - 1.96 * se), min(1.0, p + 1.96 * se)
```

#### 📝 確認問題（`ch20-q06` · Lv.2 · 正答率の標準誤差 SE = sqrt(p(1-p)/n) の手計算）

正答率 $p = 0.50$ を問題数 **$n = 100$** 問で測ったとき、標準誤差 $\text{SE} = \sqrt{\frac{p(1-p)}{n}}$ はいくつになりますか？（小数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：標準誤差の式 $\text{SE} = \sqrt{\frac{p(1-p)}{n}}$ に、正答率 $p$ と問題数 $n$ を当てはめます。
- **ヒント 2**：まず分子 $p(1-p) = 0.50 \times 0.50 = 0.25$ を計算し、問題数 $n = 100$ で割ってから平方根を求めましょう。
- **正解**：`0.05`

**考え方**：$\sqrt{0.25 / 100} = \sqrt{0.0025} = 0.05$（5%ポイント）です。95%信頼区間はおおよそ $0.50 \pm 1.96 \times 0.05 \approx [0.40, 0.60]$ と±10%もの幅があります。

1. $p(1 - p) = 0.50 \times 0.50 = 0.25$。
1. $0.25 / 100 = 0.0025$。
1. $\text{SE} = \sqrt{0.0025} = 0.05$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q06

</details>

#### 📝 確認問題（`ch20-q07` · Lv.2 · 標準誤差を半分にするには問題数を何倍にすべきか（1/√n の法則））

標準誤差 $\text{SE} = \sqrt{p(1-p)/n}$ は問題数 $n$ の平方根に反比例します。信頼区間の幅（標準誤差）を**半分（$\frac{1}{2}$）** に縮めるには、問題数 $n$ を元の**何倍**に増やす必要がありますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：標準誤差 $\text{SE}$ は問題数 $n$ の平方根 $\sqrt{n}$ に反比例します。誤差を $1/2$ にするには分母の $\sqrt{n}$ を何倍にする必要があるでしょうか？
- **ヒント 2**：$\frac{1}{\sqrt{k}} = \frac{1}{2}$、つまり $\sqrt{k} = 2$ を満たす倍率 $k$ を求めましょう。
- **正解**：`4` 倍

**考え方**：分母が $\sqrt{n}$ なので、誤差を $1/2$ にするには $n$ を $2^2 = 4$ 倍にする必要があります（例：100問 → 400問）。

1. 元の誤差 $\propto 1/\sqrt{n}$。
1. $n$ を $4n$ にすると $1/\sqrt{4n} = \frac{1}{2} \cdot (1/\sqrt{n})$。
1. よって **4倍** の問題数が必要。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q07

</details>

*(この節の追加問題 `ch20-q08` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html で自動採点・復習できます)*

自分の実験の結果（`quiz_results`）でも計算してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
diff, low, high = bootstrap_diff(quiz_results["ベースライン"], quiz_results["ボンゴLoRA"])
print(f"あなたの実験：差 {diff:+.2f}  95%信頼区間 [{low:+.2f}, {high:+.2f}]")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
あなたの実験：差 -0.05  95%信頼区間 [-0.25, +0.15]
```

</details>

#### 📝 確認問題（`ch20-q09` · Lv.2 · ブートストラップの復元抽出と「対応のある比較」の利点）

ブートストラップ法で元の $n$ 問から $n$ 問を選び直すとき、必ず**復元抽出**（同じ問題を重複して選ぶことを許す）を使う理由はどれですか？

- **a.** Python標準の `random` モジュールには重複なしで要素を抽出する関数が用意されていないから
- **b.** 非復元抽出で $n$ 問から $n$ 問選ぶと毎回まったく同じ集合になり、ばらつきが常に0になるから
- **c.** 復元抽出を使えばリサンプリングを1万回繰り返す必要がなく、1回の抽出だけで信頼区間が求まるから
- **d.** 同じ問題を重複して引き直すことで正答率が必ず100%へ近づき、信頼区間を狭められるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：手元に20枚のカードがあり、重複を許さずに（引いたカードを戻さずに）20枚すべて引いたら、手札の組み合わせは何通りできるか想像しましょう。
- **ヒント 2**：重複なしの非復元抽出で $n$ 件から $n$ 件すべてを選ぶと、順番が違うだけで毎回同じ集合（同じ平均・同じ差）になってしまいます。
- **正解**：`b`

**考え方**：復元抽出で「ある問題は2回出たり、ある問題は0回になったりする仮想テストセット」を1万回作ることで、元の母集団からテストセットを引き直したときのスコアのばらつきを近似できます。

1. 非復元抽出で $n$ 件から $n$ 件選ぶ → 常に同じ $n$ 件（平均も差も毎回同一）。
1. 復元抽出（`[rng.randrange(n) for _ in range(n)]`）→ 毎回少しずつ構成が変わる仮想セットを1万個作れる。
1. 1万個の差を並べ替え、下2.5%点と上97.5%点を取って95%信頼区間とする。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q09

</details>

#### 📝 確認問題（`ch20-q10` · Lv.4 · 【自力実装】正答率・標準誤差・95% Wald 信頼区間を計算する関数）

正解が `1`・不正解が `0` のリスト `results` を受け取り、正答率 `p`、標準誤差 `se = sqrt(p * (1 - p) / n)`、および95% Wald信頼区間の下限 `low = max(0.0, p - 1.96 * se)` と上限 `high = min(1.0, p + 1.96 * se)` をタプル `(p, se, low, high)` で返す関数 `accuracy_and_wald_ci(results)` を純Pythonで実装してください。

**ひな形コード：**
```python
import math

def accuracy_and_wald_ci(results):
    # 1. n = len(results), p = sum(results) / n
    # 2. se = math.sqrt(p * (1.0 - p) / n)
    # 3. low = max(0.0, p - 1.96 * se), high = min(1.0, p + 1.96 * se)
    # 4. return p, se, low, high
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：正答率 `p = sum(results) / n` と標準誤差 `se = math.sqrt(p * (1.0 - p) / n)` を求め、95%信頼区間 `p ± 1.96 * se` を `[0.0, 1.0]` に収めます。
- **ヒント 2**：`low = max(0.0, p - 1.96 * se)`、`high = min(1.0, p + 1.96 * se)` とクリップし、`(p, se, low, high)` の4要素タプルで返しましょう。

**解答例：**
```python
import math

def accuracy_and_wald_ci(results):
    n = len(results)
    p = sum(results) / n
    se = math.sqrt(p * (1.0 - p) / n)
    low = max(0.0, p - 1.96 * se)
    high = min(1.0, p + 1.96 * se)
    return p, se, low, high
```

**検算テスト（Colabで実行して確認）：**
```python
p1, se1, lo1, hi1 = accuracy_and_wald_ci([1] * 16 + [0] * 4)
assert abs(p1 - 0.8) < 1e-12
assert abs(se1 - math.sqrt(0.16 / 20)) < 1e-12
assert hi1 - lo1 > 0.34
p2, se2, lo2, hi2 = accuracy_and_wald_ci([1] * 320 + [0] * 80)
assert abs(p2 - 0.8) < 1e-12
assert abs(se2 - 0.02) < 1e-12
assert abs(lo2 - 0.7608) < 1e-4 and abs(hi2 - 0.8392) < 1e-4
p3, se3, lo3, hi3 = accuracy_and_wald_ci([1] * 10)
assert p3 == 1.0 and se3 == 0.0 and lo3 == 1.0 and hi3 == 1.0
print("ALL PASSED")
```

**考え方**：同じ正答率 80% でも、$n=20$ では信頼区間の幅が 0.35 以上もあるのに対し、$n=400$ では 0.078 まで引き締まることが自分で書いた関数で確認できます。

1. `p = sum(results) / len(results)` で正答率を計算する。
1. `se = math.sqrt(p * (1.0 - p) / len(results))` で標準誤差を計算する。
1. `p ± 1.96 * se` を `[0.0, 1.0]` に丸め込んで返す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q10

</details>

*(この節の追加問題 `ch20-q11` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html で自動採点・復習できます)*

## 評価4：LLMに審査させる（LLM-as-a-judge）

「説明が分かりやすいか」「面白いか」のような、ルールで数えにくい質は、人間が読んで判断するのが一番です。でも、毎回人間が何百件も読むのは大変です。そこで、**LLMに審査員をやってもらう**方法がよく使われます（LLM-as-a-judge）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def judge(question, answer_a, answer_b):
    prompt = (f"質問：{question}\n\n回答A：{answer_a}\n\n回答B：{answer_b}\n\n"
              "高校生にとって、分かりやすく楽しい回答はどちらですか？「A」か「B」の1文字だけで答えてください。")
    inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = baseline.generate(**inputs, max_new_tokens=2, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()[:1]

wins_ab = [judge(q, a[0], b[0]) for q, a, b in zip(test_questions, answers["ボンゴLoRA"], answers["ベースライン"])]
wins_ba = [judge(q, b[0], a[0]) for q, a, b in zip(test_questions, answers["ボンゴLoRA"], answers["ベースライン"])]
print("ボンゴを「A」に置いたとき、ボンゴの勝ち：", wins_ab.count("A"), "/", len(test_questions))
print("ボンゴを「B」に置いたとき、ボンゴの勝ち：", wins_ba.count("B"), "/", len(test_questions))
both = sum(x == "A" and y == "B" for x, y in zip(wins_ab, wins_ba))   # どちらの順番でも勝った数
print("どちらの順番でもボンゴの勝ち：", both, "/", len(test_questions))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ボンゴを「A」に置いたとき、ボンゴの勝ち： 10 / 12
ボンゴを「B」に置いたとき、ボンゴの勝ち： 5 / 12
どちらの順番でもボンゴの勝ち： 5 / 12
```

</details>

> **💡 理論のポイント：審査員にもクセがある**
>
> 同じ組み合わせなのに、並べる順番を入れ替えただけで、勝ち数が大きく変わりました。小さなLLMの審査員は「最初に見た回答（A）を選びがち」という **位置の偏り**（position bias）を持っていることが多いのです。ほかにも、「長い回答を好む」「自分と似た書き方を好む」などの偏りが知られています。
>
> 対策は、
>
> - **順番を入れ替えて2回審査**し、両方で勝ったときだけ勝ちとする（上のように）
> - 審査員には、評価されるモデルより**大きく賢いモデル**を使う
> - 審査の基準（**ルーブリック**）を具体的に書く
> - 一部を**人間も審査**して、LLM審査員と人間の判断が一致するか確かめる
>
> LLM審査は便利ですが、審査員が0.5Bでは心もとない。本格的にやるなら、第19章の4ビット読み込み（QLoRAの節）で、7Bなどの大きめのモデルを読み込んで審査員にしましょう。

<details><summary><b>基礎の補足：LLM-as-a-judge の3大バイアスと対策まとめ</b>（クリックで開く）</summary>

| バイアスの種類 | 起きる現象 | 具体的な対策 |
|---|---|---|
| **位置の偏り**（Position Bias） | 先に出された回答A（または後のB）を無意識に選びがち | A/Bの順番を入れ替えて2回審査し、**両順で勝った場合のみ勝ち**とする |
| **長さの偏り**（Verbosity Bias） | 中身が薄くても長く丁寧に見える回答を高く採点しがち | ルーブリックに「正確さと簡潔さ」を明記し、冗長な回答を減点する |
| **自己選好**（Self-Enhancement） | 審査員モデル自身と同じモデル系列の文体を好みがち | 評価対象より大きく別系統の審査モデルを使い、人間評価と一致率を照合する |

</details>

#### 📝 確認問題（`ch20-q12` · Lv.2 · LLM-as-a-judge の位置の偏り・長さの偏りと対策）

LLM-as-a-judge（LLMによる審査）で、同じ2つの回答を `(回答A, 回答B)` の順と `(回答B, 回答A)` の順に入れ替えて2回審査する理由はどれですか？

- **a.** AとBの順番を入れ替えても必ず完全に同じ判定が出ることが数学的に保証されており、動作確認のため2回回している
- **b.** LLM審査員には提示順に引きずられる位置の偏りがあるため、両順で勝った場合だけ採用し偏りを抑えるため
- **c.** 1回目の審査推論でGPUのKVキャッシュに残った古い文脈を完全に消去し、2回目の計算を正常化するため
- **d.** 同じ12問を2回実行して評価サンプルを合計24問へ増やすことで、統計的な問題数を水増しして信頼区間を狭めるため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：本文の実験で、同じモデルの回答をA側に置いたときとB側に置いたときで勝率がどう変化したか思い出しましょう。
- **ヒント 2**：A配置で `10/12` 勝ったのにB配置では `5/12` に減ったように、提示順序による偏り（position bias）をどう打ち消すかがポイントです。
- **正解**：`b`

**考え方**：LLM審査員は人間と同じく「先に出たほうを好む（位置の偏り）」「長い文章を好む（長さの偏り）」「自分と同系統の文体を好む（自己選好）」などのクセを持ちます。順序入替＋明確なルーブリック＋大型審査モデル＋人間の一部照合が定石です。

1. 位置の偏り：`judge(q, bongo, base)` と `judge(q, base, bongo)` で判定が変わる。
1. 対策：両順で勝ったものだけカウントし、片方のみなら引き分け扱いとする。
1. さらに大きな審査モデル・具体的採点基準（ルーブリック）・人間との一致率検証を組み合わせる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q12

</details>

## 評価の落とし穴

> **⚠️ つまずき注意：データ漏洩（コンタミネーション）**
>
> 評価に使う問題が、学習データに混ざっていると、モデルは「解いた」のではなく「覚えていた」だけになります。これを **コンタミネーション**（contamination、汚染 / リーク）と言います。
>
> - **自分のデータ**：学習用と評価用に同じ問題が入っていないか、データを作る最初の段階で厳密に分ける
> - **公開ベンチマーク**：有名なベンチマーク問題（MMLUなど）は、ネット上に答え付きでたくさん転載されているため、数兆トークンの事前学習データに丸ごと混入してしまい、暗記で満点を取ってしまう現象が問題視されています。そのため、定期的に非公開の問題に入れ替えたり、人間が2つのモデルの回答をブラインド比較して勝敗を決める「Chatbot Arena（LMSYS）」の **Elo系のレーティング（Elo rating）** のような仕組みが重視されるようになっています。
>
> 第14章で「トークナイザーを検証データ込みで作ったのは少しずる」と書いたのも、このデータの漏洩（リーク）の仲間です。

> **💡 理論のポイント：グッドハートの法則**
>
> 「指標が目標になると、それは良い指標ではなくなる」。経済学者グッドハートの名前で呼ばれる法則です。「！を使う割合」を目標にしてモデルを鍛えれば、「！」を連打するだけのモデルができるでしょう。ベンチマークの点数を上げること自体が目的になると、本当に欲しかった能力から離れていきます。
>
> だから評価は、**複数の方法を組み合わせ**、最後は**実際に使う場面で人間が確かめる**のが基本です。

#### 📝 確認問題（`ch20-q13` · Lv.2 · データ漏洩（コンタミネーション）とグッドハートの法則）

公開ベンチマーク（MMLUなど）の評価で問題になっている「データ漏洩（コンタミネーション）」の説明として正しいものはどれですか？

- **a.** 公開されている評価用データセットの質問文に、わずかな誤字脱字や記号ミスが紛れ込んでいる状態
- **b.** 保存されたモデルの重みファイルが転送中に破損し、全パラメータが0へ書き換わってしまう現象
- **c.** 評価バッチの実行中にGPUメモリが不足し、やむを得ずバッチサイズを1へ下げざるを得なくなる現象
- **d.** ベンチマーク問題と答えが事前学習データへ混入し、推論力ではなく暗記で高得点を取ってしまう現象

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：英語の contamination は「汚染・異物の混入」という意味です。
- **ヒント 2**：Web全体を事前学習するLLMで、公開されている有名なベンチマーク問題集自体が学習コーパスに混ざると何が起きるでしょうか？
- **正解**：`d`

**考え方**：Web全体を事前学習する現代のLLMでは、有名な公開テストが学習コーパスに紛れ込みやすいため、非公開問題への定期更新や人間ブラインド比較（Chatbot Arena の Elo レーティング等）が重視されています。

1. 公開ベンチマークがネット上に拡散 → 事前学習データに混入。
1. 未知の実力ではなく「問題集の丸暗記」でスコアが跳ね上がる。
1. 対策：学習前の厳密な分離、非公開評価セット、ブラインド人間比較。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q13

</details>

## 評価レポートを書く

最後に、結果を表にまとめます。研究でも、仕事でも、評価は「表」で語ります。下のコードの数字は、この教科書の例です。**あなたの結果の数字に書き換えて**から実行しましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
report = [
    ("保留データのパープレキシティ（低いほど良い）", "19.1", "9.2"),
    ("「！」を使う割合", "1/12", "12/12"),
    ("きちんと止まる割合", "9/12", "12/12"),
    ("4択クイズ（20問）", "17/20", "16/20（差の95%CI [-0.25, +0.15]）"),
    ("LLM審査（順番入替の両方で勝ち）", "—", "5/12"),
]
print(f"{'指標':34s} {'ベースライン':10s} ボンゴLoRA")
for metric, a, b in report:
    print(f"{metric:30s} {a:12s} {b}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
指標                                 ベースライン     ボンゴLoRA
保留データのパープレキシティ（低いほど良い）         19.1         9.2
「！」を使う割合                       1/12         12/12
きちんと止まる割合                      9/12         12/12
4択クイズ（20問）                     17/20        16/20（差の95%CI [-0.25, +0.15]）
LLM審査（順番入替の両方で勝ち）              —            5/12
```

</details>

（表の数値は出力例のものです。あなたの実験の数値で書き直してください。）

結論はこう書けます。「ボンゴLoRAは、口調と止まり方を大きく改善した（パープレキシティ・自動チェック）。一般知識の4択では1問減ったが、20問では統計的に区別できず、忘却の有無を判断するには問題数を増やす必要がある。LLM審査は位置の偏りが大きく、参考程度とする」。**言えることと、言えないことを分けて書く**。これが誠実な評価レポートです。

> **🔬 やってみよう：問題を増やしてみよう**
>
> `mini_quiz.jsonl` を真似して、4択クイズを自分で30問追加してみましょう（基本情報の過去問の知識問題などがおすすめ）。50問にすると、信頼区間はどれくらい狭くなるでしょう？ 予想してから試してください。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **毎回結果が変わる**：生成に `do_sample=False`（greedy）を使い、ブートストラップの乱数の種を固定しましょう。評価は「同じ条件で何度やっても同じ結果」が理想です。
> - **4択の確率がおかしい**：選択肢の前の空白やかぎかっこの有無で、トークンの区切りが変わります。全部の選択肢で同じ書き方にそろえましょう。
> - **審査員が「A」「B」以外を答える**：小さなモデルでは起きがちです。回答の最初の1文字だけを見る、決まった形式で答えるようfew-shotの例を入れる、などで対処します。

#### 📝 確認問題（`ch20-q14` · Lv.3 · 言えることと言えないことを分けて誠実な評価結論を書く）

本文の評価レポート（保留PPL `19.1 → 9.2`、「！」使用 `1/12 → 12/12`、停止率 `9/12 → 12/12`、20問4択 `17/20 → 16/20`［差の95%CI `[-0.25, +0.15]`］、0.5BのLLM審査 `A配置 10/12・B配置 5/12`）をもとに、**「この実験から言い切れること」** と **「まだ言い切れないこと（および次のアクション）」** に分けて誠実な結論をまとめてください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①保留PPLと自動チェックから確実に言える改善点、②20問4択の95%信頼区間が示す留保、③0.5B LLM審査のバイアスと対策、の3軸で整理しましょう。
- **ヒント 2**：「口調と停止作法は改善した」と言い切れる根拠と、「壊滅的忘却の有無はCIが0をまたぐため問題数を増やす必要があり、0.5B審査は位置の偏りが大きく大型審査員や人間評価が要る」という限界を分けて書きましょう。
- **解答の目安**：【言い切れること】保留データのパープレキシティが19.1から9.2へ大きく改善し、自動チェックでも「！」使用（1/12→12/12）と停止率（9/12→12/12）が明確に向上したため、ボンゴ先生の口調と停止の作法は確実に学習できたと言えます。
【言い切れないことと次のアクション】①20問の4択クイズは17/20から16/20へ1問減ったものの、差の95%信頼区間が[-0.25, +0.15]と0を大きくまたいでおり、壊滅的忘却が起きたとも起きていないとも言い切れません（数百問規模への問題数増加が必要）。②0.5BモデルによるLLM審査は順番入替で10/12から5/12へ大きく変動し位置の偏りが強いため参考程度にとどめ、より大きな審査モデルや人間評価で再検証する必要があります。

**考え方**：良い結果だけを強調するのではなく、サンプルサイズ不足や審査バイアスによる限界を明記することで、初めて科学的に信頼される評価レポートになります。

1. 明確な改善：保留PPL低下＋自動チェック（「！」・停止率）の大幅改善。
1. 統計的留保：20問4択の95%CIが0をまたぐ → 忘却の有無は問題数を増やして検証が必要。
1. 審査の限界：0.5B審査員の位置バイアスが大きいため、大型審査員や人間評価へ改善する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html#ex-ch20-q14

</details>

**✅ できたらチェック**

- [ ] 評価用のデータを、学習の前に取り分けた
- [ ] 保留データのパープレキシティで、口調を学んだことを確かめた
- [ ] ルールによる自動チェックを作った
- [ ] 4択クイズを選択肢の確率で解かせた
- [ ] ブートストラップで差の95%信頼区間を計算し、「言い切れるか」を判断した
- [ ] LLM審査で位置の偏りを確かめた
- [ ] 言えることと言えないことを分けて、評価の結論を書いた

> **📝 まとめ**
>
> - 評価は**保留データ**で。複数の方法（パープレキシティ・自動チェック・ベンチマーク・LLM審査・人間）を組み合わせる
> - 選択式は**選択肢の確率**で解かせる
> - 少ない問題数の差は偶然かもしれない。**ブートストラップ**で**信頼区間**を出し、0を含むなら言い切らない
> - LLM審査員には**位置の偏り**などのクセがある
> - **コンタミネーション**と**グッドハートの法則**に注意。言えることと言えないことを分けて書く

**ボンゴ先生**：第5部、完走！ キミはもう、モデルを借りて、鍛えて、正しく測れる。次の第6部では、LLMを「部品」にして、もっと大きなシステムを作る。最初は **RAG**。LLMに教科書を持たせて、カンニングOKの試験を受けさせる方法だ。ハルシネーション（もっともらしい嘘）を減らす、一番実用的な技だよ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html
- **次の章**：[第21章 資料を読んで答える：RAG](https://sakunokyumei.github.io/llm-research-textbook/21-rag.html)